# Глава 1. Введение в RAG-системы

## 1.1. Что такое RAG и зачем он нужен

Проблема LLM - её знания ограниченны данными, на которых ее обучили.

**RAG (Retrieval-Augmented Generation)** - дополненная поиском генерация - архитектурный подход, который соединяет LLM с внешней, постоянно обовляемой базой знаний.

LLM страдают от нескольких критических ограничений:
- **Галлюцинации** - когда модель сообшает ложную информацию, пытается дать ответ на любой вопрос, даже если у неё нет достоверных данных по теме;
- Устаревание данных - модель обучена на данных до 2023 года, она не знает о событиях 2024 года;
- Отсутствие спициализированных знаний. Общие модели хорошо справляются с популярными темами, но могут давать поверхностные или неточные ответы по узкоспециализированным вопросам: внутренним процессам компании, отраслевым регламентам, техническим спецификациям оборудования.

RAG дает возможность динамически обращаться к внешним источникам в момент формирования ответа:
- Когда поступает запрос пользователя, RAG‑система сначала преобразует его в векторное представление и ищет наиболее релевантные фрагменты в предварительно проиндексированной базе знаний;
- Найденная информация передаётся языковой модели как дополнительный контекст, на основе которого формируется итоговый ответ.

 ---

**RAG дополняет промпт** (или, точнее, контекст, в котором работает модель).

Вот как это работает на практике:

1.  **Без RAG:** Модель отвечает только на основе того, чему она научилась во время обучения. Промпт состоит только из инструкции и вопроса пользователя.
    *   *Запрос:* «Какие планы у компании X на следующий квартал?»
    *   *Ответ модели:* «Простите, у меня нет актуальных данных (знания заканчиваются на дату...).»

2.  **С RAG:** В исходный промпт вклинивается дополнительный блок — добытый релевантный контекст.
    *   *Сценарий:* Система нашла в базе внутренний документ о планах.
    *   *Фактический промпт, который получает модель:*
        > «Ответь на вопрос пользователя, используя **ТОЛЬКО** приведенный ниже контекст. Если в контексте нет ответа, скажи, что не знаешь.
        >
        > **Контекст:** "Согласно внутренней дорожной карте, компания X планирует запустить новую линейку продуктов в 3-м квартале."
        >
        > **Вопрос пользователя:** Какие планы у компании X на следующий квартал?»

**Главное отличие от обычного дополнения промпта:**
Обычный промпт — это статическая инструкция от разработчика. RAG-контекст — это **динамические данные**, которые извлекаются из базы знаний в реальном времени специально под конкретный запрос.

Таким образом, LLM получает не просто «дополненный промпт», а **собранный на лету контекстный пакет**, что и позволяет ей давать актуальные, фактологически точные ответы без переобучения модели.

---

Ключевое преимущество — разделение хранения знаний и их обработки. Мы можем обновлять базу знаний в режиме реального времени, добавляя новые документы, корректируя устаревшие данные, интегрируя отраслевую специфику — и всё это без необходимости переобучения LLM.

RAG — не панацея от всех проблем языковых моделей. Качество ответов напрямую зависит от качества данных в базе знаний. Если исходные документы содержат противоречивую или неточную информацию, система может усиливать эти проблемы.

RAG также не полностью устраняет галлюцинации — модель всё ещё может творчески интерпретировать найденную информацию или заполнять пробелы собственными «додумываниями». Именно поэтому современные RAG‑системы дополняются механизмами контроля качества и проверки фактов.

## 1.2. Эволюция от поисковых систем

Первое поколение поисковых технологий базировалось на принципе прямого соответствия слов. Подход TF‑IDF (Term Frequency‑Inverse Document Frequency), появившийся в 1970‑х годах, стал фундаментом информационного поиска на десятилетия вперёд. Его гениальность заключалась в простоте: часто встречающиеся в документе слова получают высокий вес, а редкие в коллекции термины считаются более значимыми.

TF‑IDF имел критический недостаток — проблему масштабирования по длине документов. Два документа разной длины с одинаковым содержанием получали различные оценки релевантности, что искажало результаты поиска. 

Алгоритм BM25 (Best Matching 25) -  ввёл концепцию насыщения частоты терминов — идею о том, что увеличение количества вхождений слова в документ даёт убывающую отдачу. Если в статье о Ломоносове его имя упоминается 12 раз вместо шести, это не делает статью в два раза более релевантной — закон убывающей полезности действует и в информационном поиске.

Семантическая революция началась с появлением искусственных нейронных сетей, способных преобразовывать слова в числовые векторы — эмбеддинги.

Технологии Word2Vec, а позднее BERT и их наследники позволили кодировать смысловое содержание текста в многомерном пространстве, в котором семантически близкие понятия располагаются рядом.

Векторный поиск кардинально изменил правила игры. Вместо поиска точных совпадений система начала искать семантическое сходство между запросом и документами. Запрос «найти информацию о покупке автомобиля» мог найти документы со словами «приобретение», «машина», «транспорт» — поисковая система научилась понимать смысл, а не только буквы.

Ключевым преимуществом стала способность поисковых систем работать с запросами на разговорном языке. Пользователь мог задать вопрос «Как лучше выбрать подержанную машину?» и получить релевантные результаты, даже если в найденных документах не было точного совпадения фразы.

Векторный поиск превосходно работает с концептуальными и абстрактными запросами, но может «терять» точные совпадения имён, артикулов, специальных терминов.

Современные системы используют гибридный подход, комбинируя алгоритм BM25 для точных лексических совпадений с векторным поиском для семантического понимания. Такая архитектура обеспечивает как точность в поиске конкретной информации, так и гибкость в понимании пользовательских интенций.

Следующий эволюционный скачок произошёл с появлением больших языковых моделей и концепции RAG. Если традиционные поисковые системы могли только найти и ранжировать документы, то RAG‑системы способны понять найденную информацию и синтезировать осмысленный ответ.

RAG объединил три ключевых компонента:
- продвинутый поисковый движок (часто гибридный);
- базу знаний с возможностью динамического обновления;
- генеративную языковую модель, способную создавать связные ответы на основе найденной информации.

Это позволило перейти от парадигмы «найти документы» к парадигме «получить ответ».

Современные RAG‑системы представляют собой полноценные интеллектуальные платформы, способные не только искать информацию, но и рассуждать, анализировать противоречия, учитывать контекст предыдущих взаимодействий. Они могут работать с мультимодальным контентом — текстами, изображениями, структурированными данными, предоставляя пользователю единый интерфейс для работы с разнородной информацией.

Ключевое отличие современных RAG‑систем от классических поисковых машин заключается в способности к диалоговому взаимодействию. Система не только находит ответ на прямой вопрос, но может уточнить неясные моменты, предложить альтернативные интерпретации запроса, объяснить свою логику.

## 1.3. Базовые принципы работы RAG

В основе любой RAG‑системы лежит идея разделения задач: 
- поиск релевантной информации;
- интеллектуальная обработка.

RAG‑система состоит из двух ключевых компонентов: 
- модуля поиска (retriever);
- модуля генерации (generator).

Эта архитектура решает сразу несколько проблем: 
- специализация позволяет оптимизировать каждый компонент под свою задачу;
- разделение обеспечивает гибкость и возможность независимого улучшения каждой части системы.

**Модуль поиска** специализируется на понимании смысла запроса и нахождении релевантных фрагментов информации в базе знаний. Его задача — не просто найти документы с совпадающими ключевыми словами, а идентифицировать контент, семантически связанный с запросом пользователя.

**Модуль генерации** берёт найденную информацию и синтезирует связный ответ, адаптированный под конкретный контекст запроса.

Прежде чем RAG‑система сможет отвечать на запросы, необходимо подготовить базу знаний в форме, оптимизированной для семантического поиска:
1. **Сегментирование** (разбивка документов). Длинные документы разбиваются на логически связанные фрагменты — чанки (chunk), каждый из которых должен содержать завершённую мысль или концепцию. Размер чанков критически важен: слишком маленькие не содержат достаточного контекста, слишком большие могут не поместиться в окно внимания языковой модели.
2. **Векторизация** - превращает текстовые фрагменты в числовые представления — эмбеддинги. Модели эмбеддинга, такие как BERT или более современные решения, анализируют семантическое содержание текста и кодируют его в многомерный вектор. Ключевое свойство эмбеддингов — семантически близкие тексты получают похожие векторные представления, что делает возможным поиск по смыслу, а не только по точным совпадениям слов.

Когда пользователь задаёт вопрос, тот же процесс векторизации применяется и к запросу. Полученный вектор запроса сравнивается с векторами всех чанков в базе знаний с помощью метрик сходства — чаще всего косинусного расстояния. Система находит `K` наиболее похожих фрагментов (обычно от 3 до 10), которые считаются наиболее релевантными для ответа на вопрос.

Критически важно понимать: векторный поиск работает с семантическим сходством, а не с лексическим совпадением. Запрос «Как починить автомобиль?» может найти документы со словами «ремонт», «машина», «техническое обслуживание» — система понимает связь между концепциями, даже если точные слова не совпадают.

**Переранжирование (reranking)** — повторное ранжирование найденных фрагментов. Специализированные модели‑реранкеры анализируют найденные чанки в контексте конкретного запроса и могут изменить их порядок, более точно оценив релевантность. Это особенно важно, когда векторный поиск находит семантически близкие, но не совсем подходящие фрагменты.

На финальном этапе найденные фрагменты объединяются с исходным запросом и передаются языковой модели. Модель получает не просто вопрос пользователя, а расширенный контекст, включающий релевантную информацию из базы знаний. Примерный промпт выглядит так:

```
Контекст: [найденные фрагменты]
Вопрос: [запрос пользователя]
Ответь на вопрос, основываясь на предоставленном контексте.
```

Языковая модель использует как свои параметрические знания, так и предоставленный контекст для формирования ответа. Это решает проблему устаревания знаний — модель может опираться на актуальную информацию из внешних источников, даже если эти данные появились после её обучения.

RAG превращает статичную языковую модель в систему с динамической, обновляемой памятью. Знания больше не «зашиты» в параметрах модели, а хранятся во внешней базе знаний, которую можно обновлять, расширять и модифицировать без перетренировки модели. Это создаёт непараметрическую память — знания, существующие независимо от весов нейронной сети.

Современные реализации RAG выходят за рамки простой схемы «один запрос — один поиск — один ответ». Системы с цепочкой поиска могут выполнять несколько итераций поиска, уточняя запросы на основе промежуточных результатов. Агентные RAG‑архитектуры используют языковую модель для принятия решений о том, какие функции поиска вызывать и как обрабатывать результаты.

Некоторые системы реализуют иерархический поиск: сначала грубый поиск по большим фрагментам, затем точный поиск внутри выбранных разделов. Это позволяет балансировать между эффективностью (скоростью поиска) и точностью (качеством найденной информации).

Ключевое преимущество RAG заключается в способности адаптироваться к контексту запроса. RAG‑система может находить различные аспекты информации в зависимости от того, как сформулирован запрос. Один и тот же документ может быть релевантен для разных вопросов, но RAG будет извлекать из него разную информацию в зависимости от контекста.

## 1.4. Минимальный практический пример на Python

Библиотека `sentence-transformers` будет генерировать эмбеддинги, `FAISS` послужит векторной базой данных, а `gigachain` — языковой моделью для генерации ответов.

In [1]:
pip install gigachain langchain-community faiss-cpu

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
from typing import List
from langchain_community.embeddings import GigaChatEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.llms import GigaChat
from langchain.chains import RetrievalQA
from langchain.schema import Document

class SimpleRAG:
    def __init__(self, api_key: str, scope: str = "GIGACHAT_API_PERS"):
        """
        Инициализация RAG-системы с GigaChat.
        
        :param api_key: Токен доступа к GigaChat API
        :param scope: Версия API (по умолчанию для физлиц)
        """
        # Инициализируем модель эмбеддингов GigaChat
        self.embeddings = GigaChatEmbeddings(
            credentials=api_key,
            scope=scope,
            verify_ssl_certs=False
        )
        
        # Инициализируем языковую модель GigaChat
        self.llm = GigaChat(
            credentials=api_key,
            scope=scope,
            verify_ssl_certs=False,
            temperature=0.1,
            max_tokens=200
        )
        
        # Хранилище для документов и индекс FAISS
        self.documents: List[Document] = []
        self.vectorstore = None

    def add_documents(self, documents: List[str]):
        """
        Добавляем документы в базу знаний.
        
        :param documents: Список текстов документов
        """
        # Преобразуем строки в объекты Document
        new_docs = [Document(page_content=text) for text in documents]
        self.documents.extend(new_docs)

        if self.vectorstore is None:
            # Создаём новое векторное хранилище
            self.vectorstore = FAISS.from_documents(
                documents=self.documents,
                embedding=self.embeddings
            )
        else:
            # Добавляем документы в существующее хранилище
            self.vectorstore.add_documents(new_docs)

    def search(self, query: str, k: int = 3) -> List[str]:
        """
        Ищем наиболее релевантные документы.
        
        :param query: Поисковый запрос
        :param k: Количество возвращаемых документов
        :return: Список текстов найденных документов
        """
        if self.vectorstore is None:
            return []
        
        # Поиск с использованием similarity_search
        docs = self.vectorstore.similarity_search(query, k=k)
        return [doc.page_content for doc in docs]

    def generate_answer(self, query: str, context: List[str]) -> str:
        """
        Генерируем ответ на основе найденного контекста.
        
        :param query: Вопрос пользователя
        :param context: Список документов контекста
        :return: Сгенерированный ответ
        """
        # Формируем промпт с контекстом
        context_text = "\n".join(
            [f"Документ {i+1}: {doc}" for i, doc in enumerate(context)]
        )
        
        prompt = f"""Контекст:
{context_text}

Вопрос: {query}

Ответь на вопрос, используя только информацию из предоставленного контекста.
Если ответа нет в контексте, так и напиши."""

        # Вызываем GigaChat API
        response = self.llm.invoke(prompt)
        return response.strip()

    def ask(self, query: str) -> dict:
        """
        Основная функция: поиск + генерация ответа.
        
        :param query: Вопрос пользователя
        :return: Словарь с запросом, ответом и источниками
        """
        # Находим релевантные документы
        relevant_docs = self.search(query)

        # Генерируем ответ
        answer = self.generate_answer(query, relevant_docs)

        return {
            "query": query,
            "answer": answer,
            "sources": relevant_docs
        }

Создадим базу знаний и протестируем систему:

In [ ]:
# Создаём экземпляр RAG‑системы
rag = SimpleRAG(api_key="your-openai-api-key")

# Добавляем документы в базу знаний
documents = [
    "Python — высокоуровневый язык программирования общего назначения. Создан в 1991 году Гвидо ван Россумом.",
    "RAG (Retrieval‑Augmented Generation) — архитектурный подход, который объединяет поиск информации с генерацией текста.",
    "Векторные базы данных хранят данные в виде высокомерных векторов и позволяют выполнять семантический поиск.",
    "FAISS — библиотека для эффективного поиска схожих векторов.",
    "Эмбеддинги — это векторные представления текста, которые кодируют семантическое содержание в числовой форме."
]

rag.add_documents(documents)

# Задаём вопросы системе
questions = [
    "Кто создал Python?",
    "Что такое RAG?",
    "Как работают векторные базы данных?"
]

for question in questions:
    result = rag.ask(question)
    print(f"Вопрос: {result['query']}")
    print(f"Ответ: {result['answer']}")
    print(f"Источники: {result['sources']}")
    print("-" * 50)


C:\Users\user\AppData\Local\Temp\ipykernel_17548\3170654767.py:18: LangChainDeprecationWarning: The class `GigaChatEmbeddings` was deprecated in LangChain 0.3.5 and will be removed in 1.0. An updated version of the class exists in the langchain-gigachat package and should be used instead. To use it run `pip install -U langchain-gigachat` and import as `from langchain_gigachat import GigaChatEmbeddings`.
  self.embeddings = GigaChatEmbeddings(


Вопрос: Кто создал Python?
Ответ: Python был создан **Гвидо ван Россумом** в 1991 году.
Источники: ['Python — высокоуровневый язык программирования общего назначения. Создан в 1991\xa0году Гвидо ван Россумом.', 'FAISS — библиотека для эффективного поиска схожих векторов.', 'RAG (Retrieval‑Augmented Generation) — архитектурный подход, который объединяет поиск информации с генерацией текста.']
--------------------------------------------------
Вопрос: Что такое RAG?
Ответ: На основании предоставленного контекста:

**RAG (Retrieval-Augmented Generation)** — это архитектурный подход, объединяющий два ключевых компонента:
- Поиск релевантной информации (Retrieval).
- Генерацию текста на основе найденной информации (Generation).

Таким образом, RAG позволяет использовать результаты поиска (например, с помощью библиотеки FAISS) для улучшения качества и точности генерируемого текста путем добавления дополнительной информации.
Источники: ['RAG (Retrieval‑Augmented Generation) — архитектурный по

Giga generation stopped with reason: length


Вопрос: Как работают векторные базы данных?
Ответ: ## Анализ контекста

### Документ 1
- **Информация:** Векторные базы данных хранят данные в виде высокомерных (вероятно, имеется в виду высокодimensional) векторов.
- **Ключевое понятие:** Семантический поиск.

### Документ 2
- **Информация:** FAISS — библиотека для эффективного поиска схожих векторов.
- **Ключевое понятие:** Эффективный поиск похожих векторов.

### Документ 3
- **Информация:** Эмбеддинги — это векторные представления текста, которые кодируют семантическое содержание в числовой форме.
- **Ключевое понятие:** Векторное представление текста, семантическая информация.

## Итоговый вывод

На основе представленной информации можно сделать следующие выводы относительно работы векторных баз данных:

1. Векторные базы данных используют векторные представления данных высокой размерности.
2. Эти векторные представления содержат семантическую информацию, позволяющую эффективно проводить семант
Источники: ['Векторные базы данных х

Этот минимальный пример демонстрирует базовые принципы, но в реальных системах потребуются дополнительные компоненты:

*   сегментирование длинных документов на фрагменты оптимального размера;
*   предварительная обработка текста (очистка, нормализация);
*   гибридный поиск, комбинирующий семантический и лексический поиск;
*   реранкинг найденных фрагментов для повышения точности;
*   обработка ошибок и валидация входных данных.

```mermaid
flowchart TB
    %% ============ СТИЛИ ============
    classDef base fill:#fff,stroke:#333,stroke-width:1px,color:#000;
    classDef store fill:#f9f9f9,stroke:#333,stroke-width:2px,stroke-dasharray: 5 5;
    classDef llm fill:#e0e0e0,stroke:#333,stroke-width:2px;

    %% ============ ПОТОК ДАННЫХ ============

    subgraph S1 [1. ПОДГОТОВКА - Индексация]
        A[① Загрузка документов] --> B[② Очистка и предобработка]
        B --> C[③ Разбивка на чанки]
        C --> D[④ Векторизация чанков - Embedding]
        D --> E[(⑤ Сохранение в векторную БД)]
    end

    subgraph S2 [2. ЗАПРОС - Retrieval]
        F[⑥ Вопрос пользователя] --> G[⑦ Переформулировка запроса]
        G --> H[⑧ Векторизация вопроса]
        H --> I[⑨ Поиск похожих векторов в БД]
        E -.-> I
        I --> J[⑩ Извлечение Top-K чанков]
        J --> K[⑪ Реранкинг и фильтрация]
        K --> L[⑫ Формирование контекста]
    end

    subgraph S3 [3. ГЕНЕРАЦИЯ - Augmented Generation]
        L --> M[⑬ Сборка итогового промпта]
        F --> M
        M --> N[[⑭ LLM: Генерация ответа]]
        N --> O[⑮ Постобработка и цитирование]
        O --> P[⑯ Финальный ответ пользователю]
    end

    %% Альтернативный путь: нет релевантного контекста
    K -->|Нет релевантных данных| Q[⛔ Ответ: Недостаточно информации]

    %% Применение стилей
    class A,B,C,D,F,G,H,I,J,K,L,M,O,P,Q base;
    class E store;
    class N llm;
```

---

### Краткая легенда по номерам:

| Номер | Этап | Описание |
|-------|------|----------|
| ① | Загрузка | Поступление PDF, HTML, API |
| ② | Очистка | Удаление шума, HTML-тегов |
| ③ | Чанкование | Разбивка на осмысленные блоки |
| ④ | Векторизация | Текст → числа через эмбеддер |
| ⑤ | Индексация | Сохранение в Pinecone/Milvus |
| ⑥ | Вопрос | Пользователь вводит запрос |
| ⑦ | Переформулировка | Query rewriting для точности |
| ⑧ | Векторизация вопроса | Вопрос → вектор |
| ⑨ | Поиск | Сравнение с векторами в БД |
| ⑩ | Извлечение | Top-K релевантных чанков |
| ⑪ | Реранкинг | Тонкая переоценка результатов |
| ⑫ | Контекст | Сборка найденных фрагментов |
| ⑬ | Сборка промпта | Контекст + системный промпт + вопрос |
| ⑭ | LLM | GPT/Claude/Llama генерирует ответ |
| ⑮ | Постобработка | Добавление ссылок, форматирование |
| ⑯ | Ответ | Сообщение отправлено в чат |

---